# NB 3.2 &mdash; Preparar les variables per al model

**MP 5134** &mdash; UT3

*Dades: California Housing, districtes censals de Califòrnia (cens de 1990).*

---
### Què farem avui

Al NB 3.1 vam mirar les variables de California Housing i en vam treure una
llista de decisions: la zona (`ocean_proximity`) és text i el model no la pot
fer servir, els quatre totals del districte diuen gairebé el mateix, i les
persones per llar tenen uns valors extrems que enganyen. Avui **posarem en
pràctica aquestes decisions** i mirarem, a cada pas, si el model del NB 2.2
s'equivoca menys.

Quan acabis hauries de saber respondre aquestes preguntes:

- Com es converteix una columna de text en números que el model pugui fer servir?
- Què fem amb una categoria que gairebé no surt, o amb una que no hem vist mai?
- Com s'omplen els buits d'una columna?
- Quan ajuda el logaritme, i per què?
- Per a què serveix posar totes les variables a la mateixa escala?
- Per què totes aquestes eines s'ajusten **només amb l'entrenament**?

Pel camí trobaràs dues pauses **Ara tu**.

## 1. El punt de partida: el model del NB 2.2

Comencem exactament on ho vam deixar al NB 2.2: les mateixes dades, el mateix
test segellat i la mateixa partició en entrenament i validació. Les dues
`random_state=42` fan que siguin els mateixos districtes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# California Housing: districtes censals de Califòrnia (cens de 1990)
URL_DADES = ("https://raw.githubusercontent.com/ageron/handson-ml2/"
             "master/datasets/housing/housing.csv")

df = pd.read_csv(URL_DADES)

dades, test_segellat = train_test_split(df, test_size=0.2, random_state=42)
del test_segellat   # el pany del NB 2.2: no s'obre fins a la UT10

FEATURES = ["median_income", "latitude", "longitude", "housing_median_age",
            "total_rooms", "total_bedrooms", "population", "households"]
OBJECTIU = "median_house_value"

dades = dades.dropna(subset=FEATURES + [OBJECTIU])
train, valid = train_test_split(dades, test_size=0.25, random_state=42)

print(f"Districtes d'entrenament: {len(train)}")
print(f"Districtes de validació:  {len(valid)}")

Ara preparem una funció petita, `provar`, que entrena una regressió lineal amb
les columnes que li donem, en calcula el MAE d'entrenament i el de validació i
ho apunta a la llista `resultats`. Així, al final del notebook tindrem tots els
intents en una sola taula.

La provem de seguida amb el model de les vuit variables del NB 2.2, que és el
que volem millorar.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

resultats = []


def provar(nom, X_train, X_valid):
    """Entrena una regressió lineal, n'apunta els dos MAE a resultats i torna el model."""
    model = LinearRegression().fit(X_train, train[OBJECTIU])
    resultats.append({
        "model": nom,
        "MAE entrenament": mean_absolute_error(train[OBJECTIU], model.predict(X_train)),
        "MAE validació": mean_absolute_error(valid[OBJECTIU], model.predict(X_valid)),
    })
    return model


provar("vuit variables (NB 2.2)", train[FEATURES], valid[FEATURES])
pd.DataFrame(resultats).round(0)

Uns **51.200 \$** d'error de validació, el mateix número del NB 2.2. Aquest és el
llistó d'avui.

## 2. Del text als números: la codificació one-hot

La zona del districte, `ocean_proximity`, és una de les variables que més haurien
d'ajudar: ja vam veure al mapa del NB 2.2 que la costa és cara i l'interior és
barat. Però és text, i un model de scikit-learn només accepta números.

La primera idea que ve al cap és numerar les zones: `INLAND` = 0, `<1H OCEAN` = 1,
`NEAR OCEAN` = 2, i així. És una mala idea. El model entendria que hi ha un
**ordre** i unes **distàncies**: que `NEAR OCEAN` és «el doble» de `<1H OCEAN`,
o que l'interior és «a un pas» de la primera zona i «a dos» de la segona. Res
d'això té sentit, perquè les zones no tenen cap ordre. (Quan les categories sí que
tenen un ordre de veritat, numerar-les és bona idea. Ho veurem al NB 3.3.)

La solució habitual és la **codificació one-hot**: fer **una columna per a cada
categoria**, amb un 1 a la columna de la zona del districte i un 0 a totes les
altres. Scikit-learn té una fàbrica que ho fa, `OneHotEncoder`. Mirem-la amb uns
quants districtes d'entrenament.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

exemple = train[["ocean_proximity"]].head(6)

fabrica_prova = OneHotEncoder(sparse_output=False)
fabrica_prova.fit(train[["ocean_proximity"]])

pd.DataFrame(fabrica_prova.transform(exemple),
             columns=fabrica_prova.get_feature_names_out(), index=exemple.index).join(exemple)

És la mateixa idea del NB 2.3: una **fàbrica** que es prepara amb `fit` i
converteix taules amb `transform`. Aquí el `fit` aprèn quines zones existeixen
(en mira la columna de l'entrenament i en treu la llista), i el `transform` fa
les columnes. `sparse_output=False` li demana que torni una taula normal; si no
ho diem, torna un format comprimit que estalvia memòria però que no es llegeix
bé.

Cada fila té un sol 1, a la columna de la seva zona. Ara el model pot donar a
cada zona el seu propi pes, sense inventar-se cap ordre.

Però hi ha un problema que ja coneixes: la zona `ISLAND`. Quants districtes
d'aquesta zona tenim a l'entrenament?

In [ ]:
train["ocean_proximity"].value_counts()

**Dos.** Una columna sencera per a dos districtes de dotze mil és una columna
que el model no pot aprendre bé: tot el que sàpiga de les illes ho haurà tret
de dues cases. Quan una categoria surt molt poc, la solució és **agrupar-la** amb
les altres categories minoritàries en una de sola, que vol dir «altres».

`OneHotEncoder` ho fa sol amb el paràmetre `min_frequency`: les categories que
surtin menys vegades d'aquest número van a parar a una columna comuna. I, ja
posats, li diem també què ha de fer si mai li arriba una zona que no ha vist a
l'entrenament: `handle_unknown="infrequent_if_exist"` vol dir «posa-la també a la
columna dels altres». Aquesta és la fàbrica que farem servir.

In [ ]:
fabrica_zona = OneHotEncoder(min_frequency=10, handle_unknown="infrequent_if_exist",
                             sparse_output=False)
fabrica_zona.fit(train[["ocean_proximity"]])   # només amb l'entrenament


def zona(taula):
    """Converteix la columna ocean_proximity de la taula en columnes one-hot."""
    return pd.DataFrame(fabrica_zona.transform(taula[["ocean_proximity"]]),
                        columns=fabrica_zona.get_feature_names_out(), index=taula.index)


zona(train).sum()

Ara hi ha **quatre zones amb columna pròpia i una de grup**,
`ocean_proximity_infrequent_sklearn` («poc freqüent», en el nom que li posa
scikit-learn), on han anat a parar els dos districtes de les illes. La suma de
cada columna és el nombre de districtes de cada zona.

La funció `zona` és només una drecera perquè no haguem de repetir aquelles tres
línies cada vegada: rep una taula i en torna les columnes one-hot, amb la fàbrica
ja preparada. Afegim aquestes columnes a les vuit variables del NB 2.2, i a
veure què passa.

In [ ]:
provar("vuit variables + zona",
       pd.concat([train[FEATURES], zona(train)], axis=1),
       pd.concat([valid[FEATURES], zona(valid)], axis=1))

pd.DataFrame(resultats).round(0)

`pd.concat(..., axis=1)` enganxa taules una al costat de l'altra: les vuit
variables i les cinc columnes de la zona.

L'error de validació baixa uns **1.100 \$**. No és gaire, i té una explicació: el
model ja tenia la latitud i la longitud, que en bona part ja diuen si un districte
és a la costa o a l'interior. La zona hi afegeix poc de nou, però hi afegeix.

---
### Ara tu · Exercici 1: districtes que no hem vist mai

Fes que `fabrica_zona` converteixi aquesta taula de tres districtes nous:

```python
nous = pd.DataFrame({"ocean_proximity": ["INLAND", "ISLAND", "LAKE"]})
```

Mostra el resultat com una taula amb el nom de les columnes, igual que a la
cel·la del principi de la secció. On va a parar cada districte? `LAKE` no existeix
a les dades: què hauria passat si la fàbrica no tingués el paràmetre
`handle_unknown`? Prova-ho amb `fabrica_prova`, la fàbrica sense paràmetres, i
explica per què és important que una fàbrica sàpiga què fer amb una categoria
desconeguda quan el model ja és en marxa.

Pista: la funció `zona(nous)` ja ho fa tot amb `fabrica_zona`. Per a
`fabrica_prova`, fes servir `fabrica_prova.transform(nous)`. Si dona un error,
llegeix-ne l'última línia.

In [ ]:
# Escriu aquí el teu codi

## 3. Omplir els buits

Recorda el NB 2.2: `total_bedrooms` té 207 buits al fitxer, i tots van caure al
test segellat. Avui no ens molesten, però quan obrim el segell a la **UT10** els
haurem d'omplir, perquè `fit()` i `predict()` no accepten valors absents. Deixem
preparada la fàbrica que ho farà, `SimpleImputer`.

In [ ]:
from sklearn.impute import SimpleImputer

fabrica_buits = SimpleImputer(strategy="median")
fabrica_buits.fit(train[["total_bedrooms"]])   # només amb l'entrenament

print(f"Valor amb què omplirà els buits: {fabrica_buits.statistics_[0]:.0f} dormitoris")

# Una prova amb tres districtes inventats, el del mig sense dada
amb_buits = pd.DataFrame({"total_bedrooms": [350, np.nan, 1200]})
fabrica_buits.transform(amb_buits)

`strategy="median"` vol dir que omplirà cada buit amb la **mediana** de la
columna a l'entrenament: 438 dormitoris. Fem servir la mediana i no la mitjana
pel que vam veure al NB 3.1: aquesta columna té uns quants districtes enormes
que estirarien la mitjana cap amunt, i la mediana no se n'immuta.

Fixa't en la diferència amb les fàbriques del NB 2.3. `PolynomialFeatures` no
aprenia cap número de les dades. **Aquesta sí que n'aprèn un**, i és exactament el
cas que t'anunciava aquell notebook: si la preparéssim amb la validació (o, a la
UT10, amb el test), estaríem fent servir informació de districtes que se suposa
que encara no coneixem. És una **fuga d'informació**, i per això totes les
fàbriques d'avui fan el `fit` amb `train` i res més.

(Omplir els dormitoris d'un districte amb la mediana de tots és una solució
grollera: un districte petit i un d'enorme rebrien el mateix número. Per a uns
pocs buits és prou bona. N'hi ha de més fines, però no les necessitarem.)

## 4. Proporcions en lloc de totals

La segona decisió del NB 3.1 era sobre els quatre totals del districte
(`total_rooms`, `total_bedrooms`, `population` i `households`). Diuen gairebé el
mateix, la mida del districte, i ens va semblar millor substituir-los per
**proporcions** que descriguin com són les llars: les habitacions per llar, el
percentatge de dormitoris i les persones per llar. Les fabriquem igual que al
NB 2.3.

In [ ]:
for taula in (train, valid):
    taula["habitacions_per_llar"] = taula["total_rooms"] / taula["households"]
    taula["pct_dormitoris"] = 100 * taula["total_bedrooms"] / taula["total_rooms"]
    taula["persones_per_llar"] = taula["population"] / taula["households"]

BASE = ["median_income", "latitude", "longitude", "housing_median_age"]
PROPORCIONS = ["habitacions_per_llar", "pct_dormitoris", "persones_per_llar"]

provar("proporcions + zona",
       pd.concat([train[BASE + PROPORCIONS], zona(train)], axis=1),
       pd.concat([valid[BASE + PROPORCIONS], zona(valid)], axis=1))

pd.DataFrame(resultats).round(0)

Aquestes columnes no necessiten cap fàbrica: dividir una columna per una altra
és una fórmula fixa, que no aprèn res de les dades, i per tant no hi ha perill
de fuga. Es pot fer igual a l'entrenament, a la validació i al test.

`BASE` són les quatre variables que no toquem, i `PROPORCIONS`, les tres noves.
Ara el model té **set columnes numèriques en lloc de vuit**, més la zona.

I el resultat és una sorpresa: **l'error puja**, fins a uns 51.700 \$. Pitjor que
abans de començar! Què ha passat? Recorda el NB 3.1: `persones_per_llar` té uns
quants districtes amb valors absurds, de centenars de persones per llar, que
s'enduien la correlació de Pearson. Una regressió lineal té el mateix problema
que Pearson: aquells pocs punts estiren la recta cap a ells i l'espatllen per a
tots els altres.

## 5. El logaritme: domar els valors extrems

Mirem com es reparteix `persones_per_llar` amb un histograma, i al costat, com
queda si li apliquem el **logaritme**.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4))

eixos[0].hist(train["persones_per_llar"], bins=60)
eixos[0].set_title("persones_per_llar")
eixos[1].hist(np.log(train["persones_per_llar"]), bins=60)
eixos[1].set_title("log(persones_per_llar)")

for eix in eixos:
    eix.set_ylabel("Nombre de districtes")
plt.show()

print(f"Màxim de persones per llar a l'entrenament: {train['persones_per_llar'].max():.0f}")

A l'esquerra, l'histograma és una sola barra contra la paret. No és que no hi
hagi res més: és que l'eix horitzontal ha d'arribar fins a **600 persones per
llar**, el districte més extrem de l'entrenament, i tota la resta queda aixafada
en un racó. A la dreta, amb el logaritme, apareix una campana ben formada.

No cal que recordis la fórmula del logaritme. El que importa és **què fa**:
**encongeix els números grans molt més que els petits**. Mira-ho:

| persones per llar | 1 | 3 | 10 | 100 | 600 |
|---|---|---|---|---|---|
| logaritme | 0 | 1,1 | 2,3 | 4,6 | 6,4 |

Passar de 3 a 10 persones per llar (7 més) suma 1,2 al logaritme; passar de 100 a
600 (500 més!) només en suma 1,8. Els valors normals queden separats entre ells, i els extrems
queden a prop dels altres en lloc de quedar a l'altra punta del món. Per això
el logaritme és la transformació de les variables amb una **cua llarga cap a la
dreta**: ingressos, poblacions, preus, mides de fitxers…

Dues coses pràctiques. Com la divisió, el logaritme és una fórmula fixa: no
aprèn res i no té perill de fuga. I només funciona amb valors **més grans que
zero**. Si una columna té zeros, es fa servir `np.log1p`, que suma 1 abans de fer
el logaritme.

Substituïm `persones_per_llar` pel seu logaritme i tornem a provar.

In [ ]:
for taula in (train, valid):
    taula["log_persones_per_llar"] = np.log(taula["persones_per_llar"])

NUMERIQUES = BASE + ["habitacions_per_llar", "pct_dormitoris", "log_persones_per_llar"]

provar("proporcions amb logaritme + zona",
       pd.concat([train[NUMERIQUES], zona(train)], axis=1),
       pd.concat([valid[NUMERIQUES], zona(valid)], axis=1))

pd.DataFrame(resultats).round(0)

Ara sí: **uns 48.200 \$**, 3.000 \$ menys que el model del NB 2.2, i amb menys
columnes numèriques. La mateixa informació, ben preparada, val més que la
informació en brut. `NUMERIQUES` és la llista definitiva de les set columnes
numèriques del model.

---
### Ara tu · Exercici 2: el logaritme sempre ajuda?

Fes el mateix amb `habitacions_per_llar`:

- Dibuixa'n l'histograma a l'entrenament, abans i després del logaritme. Té una
  cua llarga com la de les persones per llar?
- Crea la columna `log_habitacions_per_llar` a `train` i a `valid`, i prova el
  model de la cel·la anterior amb el logaritme de les habitacions en lloc de les
  habitacions tal qual. Millora l'error de validació?
- Amb el que has vist, quan creus que val la pena aplicar el logaritme?

Pista: la llista de columnes seria
`BASE + ["log_habitacions_per_llar", "pct_dormitoris", "log_persones_per_llar"]`.

In [ ]:
# Escriu aquí el teu codi

## 6. Totes a la mateixa escala

Queda l'última eina, l'escalat. Mira els coeficients del model que acabem
d'entrenar, el de les proporcions amb logaritme: el darrer que hem provat.

In [ ]:
model_sense_escalar = LinearRegression().fit(pd.concat([train[NUMERIQUES], zona(train)], axis=1),
                                             train[OBJECTIU])

# Els set primers coeficients són els de les columnes numèriques; la resta, els de la zona
pd.Series(model_sense_escalar.coef_[:len(NUMERIQUES)], index=NUMERIQUES).round(0)

És el mateix model de la secció 5. Només en mostrem els coeficients de les set
columnes numèriques, que són els que ara ens interessen.

Quina variable pesa més? A primera vista, el logaritme de les persones per llar,
amb uns −95.000 \$, i l'antiguitat sembla que no pesi gens, amb menys de
1.000 \$. Però això és una il·lusió. Cada coeficient diu quant canvia el preu
**per cada unitat** de la seva variable, i les unitats són molt diferents: un any
d'antiguitat és un pas petit (els districtes van d'1 a 52 anys), i una unitat del
logaritme de persones per llar és un salt enorme (de 3 a 8 persones per llar,
per exemple). Comparar aquests coeficients és com comparar preus en euros i en
iens.

L'**escalat** ho resol: converteix cada columna perquè totes tinguin la mateixa
escala. La manera més habitual és l'**estandardització**, que fa la fàbrica
`StandardScaler`: a cada valor li resta la mitjana de la columna i el divideix
per la desviació (la mida típica de les diferències respecte a la mitjana).
Després d'escalar, cada columna té **mitjana 0 i desviació 1**, i un 2 vol dir el
mateix a totes: «dues vegades la diferència típica per sobre de la mitjana».

In [ ]:
from sklearn.preprocessing import StandardScaler

fabrica_escala = StandardScaler()
fabrica_escala.fit(train[NUMERIQUES])   # només amb l'entrenament


def escalar(taula):
    """Escala les columnes numèriques de la taula amb la fàbrica ja preparada."""
    return pd.DataFrame(fabrica_escala.transform(taula[NUMERIQUES]),
                        columns=NUMERIQUES, index=taula.index)


escalar(train).describe().loc[["mean", "std", "min", "max"]].round(2)

La fila `mean` és la mitjana i la fila `std`, la desviació: totes les columnes
tenen ara 0 i 1. El `fit` de `StandardScaler` aprèn **dos números per columna**,
la mitjana i la desviació de l'entrenament, i per això també es prepara només
amb `train`. Les files `min` i `max` mostren que els valors extrems hi continuen:
l'escalat canvia la unitat de mesura, però no la forma de les dades. Per domar
els extrems, el logaritme; per igualar les unitats, l'escalat.

Entrenem el model amb les columnes escalades, i la zona, que ja és de zeros i uns
i no cal escalar.

In [ ]:
model_escalat = provar("proporcions amb logaritme + zona, escalat",
                       pd.concat([escalar(train), zona(train)], axis=1),
                       pd.concat([escalar(valid), zona(valid)], axis=1))

coeficients = pd.DataFrame({
    "sense escalar": model_sense_escalar.coef_[:len(NUMERIQUES)],
    "escalat": model_escalat.coef_[:len(NUMERIQUES)],
}, index=NUMERIQUES)

print(pd.DataFrame(resultats).round(0).tail(2).to_string(index=False))
coeficients.round(0)

Dues coses a veure.

**L'error no ha canviat ni un dòlar.** Una regressió lineal s'adapta sola a
l'escala de cada variable: si una columna es divideix per deu, el seu coeficient
es multiplica per deu, i les prediccions surten iguals. Per a aquest model,
escalar no millora res.

**Però els coeficients ara sí que es poden comparar.** Cada un diu quant canvia
el preu quan la seva variable s'allunya de la mitjana una distància típica. I el
rànquing canvia molt: els ingressos passen al capdavant, amb uns 78.000 \$,
seguits de la latitud i la longitud, amb uns 60.000 \$ cadascuna. El logaritme de
les persones per llar baixa a uns 26.000 \$, i l'antiguitat passa de semblar
invisible a pesar uns 12.000 \$. (La latitud i la longitud surten totes dues
grans: és la parella que va junta del NB 3.1, i el model les fa servir
combinades per localitzar la costa.)

Llavors, per què escalem, si l'error no canvia? Per tres raons:

- **Per llegir el model**, com acabem de fer. A la **UT5** veurem una manera més
  sòlida de mesurar quines variables importen.
- **Per evitar els números enormes** que feien perdre precisió a l'ordinador al
  NB 2.3, amb els graus alts.
- I sobretot, perquè **hi ha models que no funcionen sense escalar**. El k-NN de
  la **UT4** decideix mesurant distàncies entre districtes, i si una variable va
  d'1 a 52 i una altra de 0 a 1, la primera manaria sobre totes les altres. Al
  SVM de la **UT6** li passa el mateix. Des d'ara, escalarem sempre.

## 7. Resum: totes les proves en una taula

In [ ]:
pd.DataFrame(resultats).round(0)

Llegida de dalt a baix, la taula explica el notebook:

- La **zona en one-hot** ajuda una mica, perquè la latitud i la longitud ja
  deien gran part del que diu.
- Les **proporcions** en lloc dels totals, soles, **empitjoren** el model per culpa
  dels valors extrems.
- El **logaritme** doma els extrems i deixa el millor model del notebook, uns
  3.000 \$ per sota del NB 2.2.
- L'**escalat** no canvia l'error, però fa el model llegible i és imprescindible
  per als models que vindran.

I, per sobre de tot, una regla que ja no deixarem: **tota fàbrica que aprèn
alguna cosa de les dades (`OneHotEncoder`, `SimpleImputer`, `StandardScaler`) fa
el `fit` només amb l'entrenament**, i després converteix amb `transform` la
validació i, a la UT10, el test segellat. Les fórmules fixes (dividir columnes,
el logaritme) es poden aplicar a tot arreu sense por.

Avui hem preparat cada fàbrica a mà, una per una, perquè es vegi què fa
cadascuna. A la **UT7** aprendrem a encadenar-les totes amb el model en un sol
objecte, perquè la regla del `fit` amb l'entrenament es compleixi sola.